# Análise de Eficiência Energética em Sistema de Refrigeração Industrial

Este notebook apresenta a análise exploratória do desempenho energético de um sistema de refrigeração industrial a partir de dados já tratados na camada `silver`.

O foco está na interpretação do comportamento do COP ao longo do tempo, na identificação de eventos operacionais anômalos e na relação entre eficiência energética e variáveis de processo como vazão, potência e diferença de temperatura.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

LAKEHOUSE_PATH = "/Volumes/analytics/digital_twin/data"
SILVER_PATH = f"{LAKEHOUSE_PATH}/silver"
input_file = f"{SILVER_PATH}/dados_refrigeracao_enriched.csv"

df = pd.read_csv(input_file)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# usar apenas dados fisicamente válidos
df_analysis = df.loc[~df["flag_invalido"]].copy()
df_analysis = df_analysis.sort_values("timestamp")
df_analysis.head()

## Relatório de arquivos enválidos:

In [0]:
total = len(df)
invalid = df["flag_invalido"].sum()
pct_invalid = invalid / total * 100

resumo_flags = pd.Series({
    "Total de linhas": total,
    "Linhas inválidas": invalid,
    "% inválidas": pct_invalid,
    "Vazão inválida": df["flag_vazao_invalida"].sum(),
    "Potência inválida": df["flag_potencia_invalida"].sum(),
    "ΔT inválido": df["flag_dT_invalido"].sum(),
    "Temp invertida": df["flag_temp_invertida"].sum(),
})

resumo_flags


In [0]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df_analysis["cop"], bins=40, edgecolor="black")
ax.set_title("Distribuição do COP")
ax.set_xlabel("COP (-)")
ax.set_ylabel("Frequência")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

A distribuição do COP permite avaliar a faixa predominante de operação do sistema e verificar se os valores observados estão concentrados em torno de um regime relativamente estável ou dispersos em múltiplos comportamentos operacionais.

In [0]:
cop_stats = df["cop"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
cop_stats

A análise estatística do COP resume o regime típico de operação do sistema. Média e mediana próximas sugerem comportamento global relativamente estável, enquanto os percentis ajudam a delimitar a faixa operacional predominante e a distinguir oscilações normais de eventos mais extremos.

In [0]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df_analysis["timestamp"], df_analysis["cop"], linewidth=1.0, label="COP")
ax.axhline(df_analysis["cop"].mean(), linestyle="--", linewidth=1, label="COP médio")

ax.set_title("Evolução Temporal do COP")
ax.set_xlabel("Data")
ax.set_ylabel("COP (-)")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

A série temporal do COP permite observar a estabilidade do sistema ao longo do período analisado e identificar quedas pontuais de eficiência associadas a eventos operacionais específicos.